##### ***RLHF概述（Reinforcement Learning from Human Feedback）***
###### 在前面的学习中，我们已经了解了语言模型从Pretraining到SFT的基本过程。Pretraining让模型通过预测下一个Token学习语言规律和大量知识，SFT则通过指令-回答数据让模型学会按照人类给出的示例完成任务。
###### 但仅仅能够预测下一个Token，或者模仿少量指令示例，还不能完全描述我们想要的模型行为。例如，对于同一个问题，多个回答可能都没有绝对唯一的标准答案，我们更关心回答是否有帮助、是否真实、是否安全，以及是否符合用户的意图。
###### 这时，训练目标就从“预测数据中下一个Token”进一步转向“生成更符合人类偏好的回答”。RLHF就是为了完成这一目标而提出的一套后训练方法。它利用人类对模型输出的偏好反馈，训练一个能够评价回答质量的奖励模型，再使用强化学习优化语言模型的生成策略。

##### ***RLHF在整个模型训练流程中的位置***
###### 如果把语言模型的训练过程串起来，可以得到下面的流程：
$$\text{Pretraining}\rightarrow\text{SFT}\rightarrow\text{Preference Learning}\rightarrow\text{RLHF}$$
###### Pretraining阶段主要学习语言、知识和一般能力；SFT阶段主要学习如何理解指令并组织回答；RLHF阶段则进一步调整模型的行为分布，使模型更加倾向于产生人类偏好的回答。
###### 因此，RLHF通常不是从零训练一个语言模型，也不是用来替代Pretraining。它是在已有语言模型的基础上，通过额外的偏好信号进行行为层面的优化。RLHF可以改变模型更愿意怎样回答，但它能够新增的知识通常非常有限。
###### 需要注意的是，RLHF并不等于所有对齐方法。它只是利用人类偏好反馈进行模型对齐的一条经典路线。本节先理解RLHF自身的组成，其他偏好优化方法将在后续笔记中单独介绍。

##### ***经典RLHF的整体流程***
###### 经典的RLHF流程通常可以拆分为三个主要阶段：首先对Pretraining模型进行SFT，得到一个能够基本遵循指令的模型；然后收集人类对多个回答的偏好并训练Reward Model；最后使用强化学习方法，让Policy尽可能生成能够获得高奖励的回答。
###### 这个过程可以概括为：
$$\text{Base Model}\rightarrow\text{SFT Model}\rightarrow\text{Reward Model}\rightarrow\text{RL Policy}$$
###### 在最后一个阶段，Policy会针对Prompt生成回答，Reward Model对回答进行评分，强化学习算法根据评分调整Policy的参数。为了防止Policy为了得到高分而偏离原始语言模型太远，训练中通常还会加入它与Reference Model之间的KL惩罚。
###### 这里暂时不需要记住具体的强化学习算法名称。现在只需要理解：Reward Model负责把回答质量转化为分数，强化学习阶段再利用这些分数调整Policy。具体的算法细节留到后面的笔记中。

##### ***第一阶段：监督微调（SFT）***
###### RLHF通常不会直接从一个只会续写文本的Base Model开始，而是先使用人工编写的指令-回答数据进行SFT。数据通常包含一个Prompt和一个人类认为较好的Response。
###### 例如：
```text
Prompt：解释什么是梯度下降。
Response：梯度下降是一种通过沿损失函数梯度的反方向更新参数来最小化损失的方法。
```
###### 在SFT过程中，模型仍然使用交叉熵损失预测Response中的下一个Token。与Pretraining相比，变化的重点不在于损失函数本身，而在于训练数据从普通文本变成了更符合指令格式的示范数据。
###### SFT的作用是先把模型调整到一个合理的起点。如果直接让一个只会续写文本的模型接受偏好强化学习，奖励模型和强化学习过程都可能需要面对大量低质量输出，训练会更加困难。

##### ***第二阶段：收集偏好数据***
###### SFT数据告诉模型“一个较好的回答应该是什么样”，而偏好数据则告诉模型“多个回答中哪个更好”。对于同一个Prompt，可以让当前模型生成多个候选回答，再由人工标注者进行比较和排序。
###### 最常见的数据形式是一个Prompt、一个更受偏好的回答和一个较不受偏好的回答：
```text
Prompt：如何处理用户密码？
chosen：应该使用经过安全哈希处理的密码，并配合盐值保存。
rejected：直接把密码以明文形式保存在数据库中。
```
###### 这种数据不一定要求标注者写出一个完美答案，只需要判断多个回答之间的相对好坏。因此，偏好数据可以表达帮助性、真实性、安全性、格式要求等更复杂的目标。
###### 对于LLM安全来说，这一步尤其重要，因为安全标准往往不是简单的分类标签，而是需要比较回答是否泄露敏感信息、是否被提示注入影响，以及是否在拒答和有用性之间取得平衡。

##### ***第三阶段：训练奖励模型（Reward Model）***
###### 人工标注者不会在强化学习训练的每一步都亲自给模型打分，因此我们需要训练一个Reward Model，让它学习人类偏好的判断方式。Reward Model本质上也是一个神经网络，只不过它的输出不是下一个Token的概率，而是一个表示回答质量的标量分数。
###### Reward Model通常接收一个Prompt和一条完整回答作为输入，并输出：
$$r_\phi(x,y)\in\mathbb{R}$$
###### 这里的$x$表示Prompt，$y$表示模型生成的回答，$\phi$表示Reward Model自身的参数，$r_\phi(x,y)$表示它对这条回答给出的分数。这个分数可以是正数，也可以是负数，并不要求一定位于$0$到$1$之间。
###### 需要注意的是，Reward Model学习的不是一个绝对正确答案，而是人类对回答的相对偏好。对于同一个Prompt，如果回答$y^+$比回答$y^-$更受偏好，那么我们希望模型满足：
$$r_\phi(x,y^+)>r_\phi(x,y^-)$$
###### 训练完成后，Reward Model就可以在没有人工实时参与的情况下，对Policy生成的回答给出近似的人类偏好分数。但它始终只是人类偏好的近似模型，并不等于真正客观、绝对正确的评价标准。

##### ***奖励模型的偏好损失如何理解***
###### 假设训练数据中包含同一个Prompt $x$ 对应的两个回答：$y^+$ 表示标注者更喜欢的回答，$y^-$表示标注者不太喜欢的回答。Reward Model分别计算两个分数：
$$r^+=r_\phi(x,y^+),\qquad r^-=r_\phi(x,y^-)$$
###### 我们首先计算两个回答之间的分数差：
$$\Delta r=r^+-r^-$$
###### 如果$\Delta r$为正，说明模型给更受偏好的回答打分更高；如果$\Delta r$为负，说明模型的判断反了。为了把这个分数差转换成一个类似“$y^+$应该被选中”的概率，我们使用Sigmoid函数：
$$\sigma(z)=\frac{1}{1+e^{-z}}$$
###### 因此，成对偏好损失可以写成：
$$\mathcal{L}_{RM}=-\log\sigma\left(r^+-r^-\right)$$
###### 这个公式可以逐步理解为：$r^+-r^-$表示两个回答的分数差，$\sigma$把分数差转换成模型认为$y^+$更好的概率，$-\log$则把这个概率转换为损失。当模型给$y^+$的分数明显高于$y^-$时，概率接近$1$，损失较小；当模型给$y^-$的分数更高时，概率接近$0$，损失就会变大。
###### 例如，如果$r^+=2.0$、$r^-=0.5$，那么分数差为$1.5$，$\sigma(1.5)$约为$0.82$，损失约为$0.20$；如果两个分数反过来，分数差为$-1.5$，概率约为$0.18$，损失约为$1.70$。因此，反向传播会推动Reward Model提高$y^+$的分数、降低$y^-$的分数。
###### 这里的Sigmoid输出只是用于计算偏好损失，并不意味着Reward Model最终输出的分数就是概率。Reward Model真正输出的是可以相互比较的标量分数。

##### ***第四阶段：使用强化学习调整Policy***
###### 有了Reward Model之后，Policy就可以不断针对Prompt生成回答，再由Reward Model进行评分。强化学习的目标是调整Policy的参数，使高奖励回答在未来出现的概率提高。
###### 在LLM中，一个完整回答由多个Token组成，因此每个Token都可以看作一次动作，整个回答则是一条轨迹。Reward Model通常在回答结束后给出整体奖励，强化学习算法需要把这个最终结果分配回生成过程中的多个Token。
###### 这一阶段可以直接联系前面学习的策略梯度：Policy生成一条回答，Reward Model给出奖励，算法根据回报和优势函数调整Policy，使高奖励回答未来出现的概率提高。
###### 实际训练中通常还会保留一份没有继续更新的参考模型，用来限制Policy不要为了追求奖励而偏离原来的语言能力太远；有时还会使用Value Model估计状态价值，帮助计算优势函数。这里先记住这些组件的作用，不展开具体算法。

##### ***RLHF的局限***
###### RLHF虽然能够让模型更符合人类偏好，但整个流程比较复杂：需要准备高质量偏好数据、训练Reward Model、采样回答、计算奖励和优势，还要控制Policy不要偏离原始模型太远。
###### 更重要的是，Reward Model只是对人类偏好的近似。如果Policy发现了Reward Model的漏洞，就可能生成表面上得分很高、实际上质量很差的回答，这种现象通常被称为奖励投机或奖励黑客。
###### 因此，RLHF并不是让模型自动变得绝对安全，而是把安全和有用等目标转化成可学习的反馈信号。反馈数据、奖励模型和评测方式中的偏差，都可能进一步影响模型行为。
###### 到这里，我们已经理解了RLHF为什么需要Reward Model，以及它如何把人类偏好传递给Policy。下一篇笔记再单独学习另一种更直接的偏好优化方法。